In [35]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

# --- Setup headless browser ---
options = Options()
options.add_argument("--headless=new")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# --- Programs to extract ---
usn_programs = {
    "Bachelor i ingeniørfag, dataingeniør": "https://www.usn.no/studier/studie-og-emneplaner/#/studieplan/ING2_2025_H%C3%98ST",
}

rows = []

def save_debug_snapshot(stage, study_program):
    name = study_program.lower().split(",")[0].replace(" ", "_")
    driver.save_screenshot(f"{name}_{stage}.png")
    try:
        html = driver.execute_script("return document.documentElement.outerHTML;")
        with open(f"{name}_{stage}.html", "w", encoding="utf-8") as f:
            f.write(html)
    except Exception as e:
        print(f"⚠️ Could not dump HTML: {e}")

def debug_shadow_root_logs():
    try:
        print("🔍 Debugging shadow root...")
        root_exists = driver.execute_script("return !!document.querySelector('usn-study-model');")
        print(" - usn-study-model exists:", root_exists)

        shadow_exists = driver.execute_script("return !!document.querySelector('usn-study-model')?.shadowRoot;")
        print(" - shadowRoot attached:", shadow_exists)

        header_count = driver.execute_script("""
            const root = document.querySelector('usn-study-model')?.shadowRoot;
            return root ? root.querySelectorAll('div.header').length : 0;
        """)
        print(" - headers found in shadowRoot:", header_count)
    except Exception as e:
        print("⚠️ Error during shadow DOM inspection:", e)

# --- Helper: Click header inside shadow DOM by visible text ---
def click_shadow_element_by_text(target_text, max_wait=10):
    js = f"""
        const host = document.querySelector('usn-study-model');
        if (!host || !host.shadowRoot) return null;
        const elements = Array.from(host.shadowRoot.querySelectorAll('div.header'));
        return elements.find(el => el.textContent.trim().includes("{target_text}"));
    """
    for _ in range(max_wait * 2):
        try:
            element = driver.execute_script(js)
            if element:
                driver.execute_script("arguments[0].click();", element)
                return True
        except Exception:
            pass
        time.sleep(0.5)

    print(f"❌ Could not find header '{target_text}'. Available headers:")
    try:
        headers = driver.execute_script("""
            const host = document.querySelector('usn-study-model');
            if (!host || !host.shadowRoot) return [];
            return Array.from(host.shadowRoot.querySelectorAll('div.header'))
                .map(e => e.textContent.trim());
        """)
        for h in headers:
            print(" -", h)
    except Exception as e:
        print("⚠️ Failed to list headers:", e)
    return False

# --- Loop through each program ---
for study_program, hash_url in usn_programs.items():
    print(f"\n🔍 Extracting for: {study_program}")

    # Open base page first
    driver.get("https://www.usn.no/studier/studie-og-emneplaner/")
    time.sleep(2)

    # Navigate via JS to trigger the SPA
    hash_part = hash_url.split("#")[-1]
    driver.execute_script(f"window.location.hash = '{hash_part}'")
    time.sleep(5)  # More time for SPA routing

    # Accept cookies if present
    try:
        cookie_button = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Godta alle')]"))
        )
        print("🍪 Accepting cookies...")
        cookie_button.click()
        time.sleep(1.5)
    except Exception:
        print("⚠️ No cookie banner found.")

    save_debug_snapshot("start", study_program)

    try:
        print("⏳ Waiting for 'usn-study-model' to be available...")
        wait.until(lambda d: d.execute_script("return document.querySelector('usn-study-model') !== null"))
        
        print("⏳ Waiting for shadowRoot to be attached...")
        wait.until(lambda d: d.execute_script("return !!document.querySelector('usn-study-model').shadowRoot"))

        print("⏳ Waiting for headers inside shadowRoot...")
        wait.until(lambda d: d.execute_script("""
            const root = document.querySelector('usn-study-model')?.shadowRoot;
            return root && root.querySelectorAll('div.header').length > 0;
        """))

    except Exception as e:
        print("❌ Shadow root or headers not ready!")
        debug_shadow_root_logs()
        save_debug_snapshot("failed_shadowroot", study_program)
        rows.append({
            "school": "USN",
            "study_program": study_program,
            "type": "Obligatorise_emner",
            "mandatory_subject": ""
        })
        continue

    save_debug_snapshot("after_shadowroot", study_program)

    try:
        if "ingeniørfag" in study_program.lower():
            print("➡️ Clicking 'Dataingeniør'...")
            if not click_shadow_element_by_text("Dataingeniør"):
                raise Exception("Failed to click 'Dataingeniør'")
            time.sleep(1)

            print("➡️ Clicking specialization...")
            if not click_shadow_element_by_text("Cyber physical systems Kongsberg A-vei"):
                raise Exception("Failed to click specialization")
            time.sleep(1)

            print("➡️ Clicking 'Obligatoriske emner'...")
            if not click_shadow_element_by_text("Obligatoriske emner"):
                raise Exception("Failed to click 'Obligatoriske emner'")
            time.sleep(1)

        save_debug_snapshot("after_clicks", study_program)

        print("🔎 Scraping subjects...")
        inner_html = driver.execute_script("""
            const host = document.querySelector('usn-study-model');
            return host.shadowRoot.innerHTML;
        """)
        soup = BeautifulSoup(inner_html, "html.parser")

        found_any = False
        for a in soup.select("a.studiemodell"):
            if "Obligatorisk" in a.get("data-original-title", ""):
                rows.append({
                    "school": "USN",
                    "study_program": study_program,
                    "type": "Obligatorise_emner",
                    "mandatory_subject": a.text.strip()
                })
                found_any = True

        if found_any:
            print(f"✅ Found {len([r for r in rows if r['study_program'] == study_program])} subjects.")
        else:
            print("⚠️ No mandatory subjects found.")
            rows.append({
                "school": "USN",
                "study_program": study_program,
                "type": "Obligatorise_emner",
                "mandatory_subject": ""
            })

    except Exception as e:
        print(f"❌ Failed for {study_program}: {e}")
        save_debug_snapshot("failed_click_or_scrape", study_program)
        rows.append({
            "school": "USN",
            "study_program": study_program,
            "type": "Obligatorise_emner",
            "mandatory_subject": ""
        })

    headers = driver.execute_script("""
        const host = document.querySelector('usn-study-model');
        if (!host || !host.shadowRoot) return [];
        return Array.from(host.shadowRoot.querySelectorAll('div.header'))
            .map(e => e.textContent.trim());
    """)
    print("🧩 Headers in shadow root:", headers)

driver.quit()

# --- Save to CSV ---
df = pd.DataFrame(rows, columns=["school", "study_program", "type", "mandatory_subject"])
df.to_csv("Mandatory_subjects.csv", mode="a", header=False, index=False)
print("\n📄 Finished writing USN mandatory subjects.")



🔍 Extracting for: Bachelor i ingeniørfag, dataingeniør
🍪 Accepting cookies...
⏳ Waiting for 'usn-study-model' to be available...
❌ Shadow root or headers not ready!
🔍 Debugging shadow root...
 - usn-study-model exists: False
 - shadowRoot attached: False
 - headers found in shadowRoot: 0

📄 Finished writing USN mandatory subjects.
